In [1]:

# Import necessary libraries

import pandas as pd
import numpy as np

# TensorFlow / Keras libraries for neural networks
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Evaluation tools
from sklearn.metrics import classification_report, accuracy_score

In [2]:

# Load the training and validation datasets
train_df = pd.read_csv("sent_train.csv")
valid_df = pd.read_csv("sent_valid.csv")

# Display the first rows to understand the structure
train_df.head()

,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0
3,$ESS: BTIG Research cuts to Neutral https://t....,0
4,$FNKO - Funko slides after Piper Jaffray PT cu...,0


In [ ]:
# Separate tweet text from sentiment labels

X_train = train_df["text"]
y_train = train_df["label"]

X_valid = valid_df["text"]
y_valid = valid_df["label"]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_valid))

Training samples: 9543
Validation samples: 2388


In [4]:
# Convert words into numerical tokens

# Maximum vocabulary size
max_words = 10000

# Create tokenizer
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")

# Fit tokenizer on training data
tokenizer.fit_on_texts(X_train)

# Convert tweets to integer sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_valid_seq = tokenizer.texts_to_sequences(X_valid)

In [5]:
# Making all sequences the same length

max_length = 40

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding="post")
X_valid_pad = pad_sequences(X_valid_seq, maxlen=max_length, padding="post")

print("Example padded sequence:")
print(X_train_pad[0])

Example padded sequence:
[5118  842 5119    8  386   10  937 1132    4    2    3 7765    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0]


In [6]:
# Convert labels to categorical format

y_train_cat = to_categorical(y_train, num_classes=3)
y_valid_cat = to_categorical(y_valid, num_classes=3)

print("Example encoded label:")
print(y_train_cat[0])

Example encoded label:
[1. 0. 0.]


In [7]:
# Creating the LSTM neural network architecture

model = Sequential()

# Embedding layer converts tokens into dense vectors
model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_length))

# LSTM layer captures sequential relationships in the text
model.add(LSTM(64))

# Dropout helps reduce overfitting
model.add(Dropout(0.5))

# Output layer for the 3 sentiment classes
model.add(Dense(3, activation="softmax"))

# Compile the model
model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

# Display the model architecture
model.summary()

c:\Users\Anxo\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train the model using the training dataset

history = model.fit(
    X_train_pad,
    y_train_cat,
    epochs=5,
    batch_size=32,
    validation_data=(X_valid_pad, y_valid_cat)
)

Epoch 1/5
299/299 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.6523 - loss: 0.8745 - val_accuracy: 0.7081 - val_loss: 0.8067
Epoch 2/5
299/299 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.7375 - loss: 0.6313 - val_accuracy: 0.7513 - val_loss: 0.5935
Epoch 3/5
299/299 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.7811 - loss: 0.4917 - val_accuracy: 0.7504 - val_loss: 0.5990
Epoch 4/5
299/299 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8210 - loss: 0.4218 - val_accuracy: 0.7651 - val_loss: 0.6555
Epoch 5/5
299/299 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8817 - loss: 0.3357 - val_accuracy: 0.7680 - val_loss: 0.6457
